# Crude Classification into Top Categories

In [1]:
# In your Jupyter notebook — must come before any ui.run() call
import asyncio
import nest_asyncio
nest_asyncio.apply()

# Fix incompatibility: wrap the patched run to accept loop_factory
_nest_run = asyncio.run

def _compatible_run(main, *, debug=None, loop_factory=None, **kwargs):
    return _nest_run(main, debug=debug)

asyncio.run = _compatible_run

import os
import regex as re
from functools import partial

#from zettelsortierung.DataModel import DataBas
from zettelsortierung.db import DataBase
from zettelsortierung.DataTypes import Classifiers, TopCategory, Zettel
from zettelsortierung.Visualisation import *
from zettelsortierung.ClassificationGUI import run_classification

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
connection_string = os.getenv('DATABASE_CONNECTION_STRING')
db = DataBase(connection_string)

In [3]:
texts = db.get_ocr_concat()

### Explore Patterns

In [4]:
def check_pattern(pattern: str) -> list[str]:
    counter = 0
    scan_ids: list[str] = []
    for text in texts:
        if len(re.findall(pattern, text[1])):
            print(text)
            scan_ids.append(text[0])
            counter += 1
    print(f'{counter}/{len(texts)} ({(100*counter/len(texts)):.2f}%)')
    return scan_ids


**Schwelm** bezieht sich eigentlich auf Herrn Böhmer und ist im Beiband unter WORTSCHATZ aufgeführt, tatsächlich handelt es sich aber vor allem um Lautschriftliche Belege.

**Mettingen** ist ein Mix aus Lautschrift und Wortschatz

In [5]:
scan_ids = check_pattern(r'Hfd Bi')

('A01-00004509_1', 'Frageb. 14 6 | usw. | Hfd Bi | Bisohofsh. Ott.')
('A05-00022909_1', '105.2 |     | hi | Hfd Bi')
('A07-00034175_1', 'Frageb. 14 2 b | Bischofsh. Ott. | Hfd Bi')
('A07-00034311_1', 'fetn | Hfd Bi')
('A07-00034637_1', 'Prrfr | D | D | Hfd Bi | eng e')
('A07-00035425_1', 'NWA II, 68 | Hfd Bie')
('A07-00036487_1', 'Sprichwort | Hfd Bi | Bischofsh, Ott. | ')
('A08-00038927_1', 'N =Grrrmmr | Bichofehagae Mris fues | Hfd Bi | Gosmeg  | I')
('A08-00042303_1', 'Birhpligee hrir uf | Grinrig Creger | Hfd Bi')
('A09-00044511_1', 'Nd. Wortatl. Fr. 46 | Hfd Bi')
('A11-00061953_1', '10 500 | Wo en  i ri i p g. |  | Hfd Bi')
('A14-00074217_1', 'Frageb. 1 la | ) | D | Hfd Bi | Bischofsh. Ott.')
('A14-00076243_1', 'Baader 1,3 | Hfe Huunebrock | HfA Ds | Atd Ed | olle Ohmt | olle Obend | tfd 14f | ölle Bbende | Id Sw | Aowends | ltfd tf | omde | Hfd Fd | oalle Oamde | Hfd Bi | olle ohmde | Hea 48 | olle Ohms | oila ãamda | Lus Is')
('B04-00453515_1', '1) mnlifn Sifuonin | 2) Lons [Ror

In [ ]:
k = 10
counter = 0
zettels: list[Zettel] = []
for scan_id in scan_ids:
    path  = db.get_full_path(scan_id)
    zettels.append(Zettel(path))
    counter += 1
    if counter == k:
        break

callback = partial(db.save_classification, Classifiers.GOLD)
run_classification(zettels, TopCategory, on_classify=callback)

RuntimeError: Cannot add middleware after an application has started

In [ ]:
k = 10
counter = 0
for scan_id in scan_ids:
    vis_scan(scan_id)
    counter += 1
    if counter == k:
        break

### Classify Patterns

In [ ]:
classification = {
    r'.rage|Fr.ge|Fra.e|Frag|Frgb|Frb|[^ ]RAGEB|F.AGE|FR.GE|FRA.E|FRAG|FRGB|Rheine[^vV]{4}': TopCategory.FRAGEBOGEN,  # Fragebogen allgemein → 203678 (16.81%)
    r'N ?W ?A|N.WA|N ?W ?H|NU[AH][ ,I\d]|.d\. Wort|N.\. Wort|Nd\. .ort|Nd\. W.rt|Nd\. Wo.t|Nd\. Wor.|[^ND][IJ]WA|MWA': TopCategory.EXZERPT,  # Niederdeutscher Wortatlas → 78197 (6.46%)
    r'.AADER|B.ADER|BA.DER|BAA.ER|BAAD.R|BAADE|Baa?der|.aader|B.ader|Ba.der|Baa.er|Baad.r|Baade': TopCategory.FRAGEBOGEN,  # Baader (Fragebögen, Nachlass) → 55902 (4.61%)
    r'S\. ?\d| s\. ?\d': TopCategory.EXZERPT,  # alles, was eine Seitenzahl hat → 39323 (3.25%)
    r'D ?[Ww] ?A|O ?W ?A |D ?UA |DWH|PWA|D.WA': TopCategory.EXZERPT,  # Deutscher Wortatlas → 38279 (3.16%)
    r'Arch\.|.rch\. [RP]|A.rch\. [RP]|Ar.h\. [RP]|Arc.\. [RP]|Arch. [RP]|Arch\..[RP]': TopCategory.EXZERPT,  # Archiv Recklinghausen → 25373 (2.09%)
    r'.löntrup|K.öntrup|Kl.ntrup|Klö.trup|Klön.rup|Klönt.up|Klöntr.p|Klöntru.|K.löntrup|Kl.öntrup|Klö.ntrup|Klön.trup|Klönt.rup|Klöntr.up|Klöntru.p|Köntrup|Klntrup|Klötrup|Klönrup|Klöntup|Klöntrp': TopCategory.EXZERPT,  # Klöntrup → 16884 (1.39%)
    r'Dor [Ww][lI1]': TopCategory.WORTSCHATZ,  # Dortmund Wellinghofen → 14286 (1.18%)
    r'[Ww]oeste|.oeste.[NW]|W.este.[NW]|Wo.ste.[NW]|Woe.te.[NW]|Woes.e.[NW]|Woest..[NW]': TopCategory.EXZERPT,  # Woeste Wb./N. → 12805 (1.06%)
    r'.hlen A|A.len A|Ah.en A|Ahl.n A|Ahle. A|Ahlen.A|A.hlen A|Ah.len A|Ahl.en A|Ahle.n A|Ahlen. A|Ahlen .A|Alen A|Ahen A|Ahln A|Ahle A|AhlenA': TopCategory.WORTSCHATZ,  # Ahlen J. Abeler → 12496 (1.03%)
    r'[^.] Koch|[^.]Koch|B.acht|Br.cht|Bra.ht|Brac.t|Brach.|B.racht|Br.acht|Bra.cht|Brac.ht|Brach.t|Bacht|Brcht|Braht|Bract': TopCategory.WORTSCHATZ,  # Koch in Bracht → 11875 (0.98%)
    r'.chöning|S.höning|Sc.öning|Sch.ning|Schö.ing|Schön.ng|Schöni.g|Schönin.|S.chöning|Sc.höning|Sch.öning|Schö.ning|Schön.ing|Schöni.ng|Schönin.g|Shöning|Scöning|Schning|Schöing|Schönng|Schönig': TopCategory.WORTSCHATZ,  # Schöning → 8672 (0.72%)
    r'.latenauWb|P.atenauWb|Pl.tenauWb|Pla.enauWb|Plat.nauWb|Plate.auWb|Platen.uWb|Platena.Wb|Platenau.b|PlatenauW.|P.latenauWb|Pl.atenauWb|Pla.tenauWb|Plat.enauWb|Plate.nauWb|Platen.auWb|Platena.uWb|Platenau.Wb|PlatenauW.b|PatenauWb|PltenauWb|PlaenauWb|PlatnauWb|PlateauWb|PlatenuWb|PlatenaWb|Platenaub': TopCategory.EXZERPT,  # PlatenauWb → 7787 (0.64%)
    r'.ittmar|D.ttmar|Di.tmar|Dit.mar|Ditt.ar|Dittm.r|Dittma.|D.ittmar|Di.ttmar|Dit.tmar|Ditt.mar|Dittm.ar|Dittma.r|Dttmar|Ditmar|Dittar|Dittmr|.önigssteele|K.nigssteele|Kö.igssteele|Kön.gssteele|Köni.ssteele|König.steele|Königs.teele|Königss.eele|Königsst.ele|Königsste.le|Königsstee.e|Königssteel.|K.önigssteele|Kö.nigssteele|Kön.igssteele|Köni.gssteele|König.ssteele|Königs.steele|Königss.teele|Königsst.eele|Königsste.ele|Königsstee.le|Königssteel.e|Knigssteele|Köigssteele|Köngssteele|Könissteele|Königsteele|Königsteele|Königsseele|Königsstele|Königsstele|Königssteee': TopCategory.WORTSCHATZ,  # Dittmar → 7501 (0.62%)
    r'.arstein|W.rstein|Wa.stein|War.tein|Wars.ein|Warst.in|Warste.n|Warstei.|W.arstein|Wa.rstein|War.stein|Wars.tein|Warst.ein|Warste.in|Warstei.n|Wrstein|Wastein|Wartein|Warsein|Warstin|Warsten': TopCategory.LAUTSCHRIFT,  # Warstein → 7408 (0.61%)
    r'[Ww]eil[Ww][Bb]': TopCategory.EXZERPT,  # WeilWb → 5634 (0.46%)
    r'W[mn][Ww]b': TopCategory.EXZERPT,  # Westmünsterländisches Wörterbuch → 5189 (0.43%)
    r'.orhelm|V.rhelm|Vo.helm|Vor.elm|Vorh.lm|Vorhe.m|Vorhel.|V.orhelm|Vo.rhelm|Vor.helm|Vorh.elm|Vorhe.lm|Vorhel.m|Vrhelm|Vohelm|Vorelm|Vorhlm|Vorhem': TopCategory.LAUTSCHRIFT,  # Vorhelm → 5091 (0.42%)
    r'.heine V|R.eine V|Rh.ine V|Rhe.ne V|Rhei.e V|Rhein. V|Rheine.V|R.heine V|Rh.eine V|Rhe.ine V|Rhei.ne V|Rhein.e V|Rheine. V|Rheine .V|Reine V|Rhine V|Rhene V|Rheie V|Rhein V|RheineV|.heine v|R.eine v|Rh.ine v|Rhe.ne v|Rhei.e v|Rhein. v|Rheine.v|R.heine v|Rh.eine v|Rhe.ine v|Rhei.ne v|Rhein.e v|Rheine. v|Rheine .v|Reine v|Rhine v|Rhene v|Rheie v|Rhein v|Rheinev': TopCategory.LAUTSCHRIFT,  # Rheine V. → 5035 (0.42%)
    r'.oester Börde|S.ester Börde|So.ster Börde|Soe.ter Börde|Soes.er Börde|Soest.r Börde|Soeste. Börde|Soester.Börde|Soester .örde|Soester B.rde|Soester Bö.de|Soester Bör.e|Soester Börd.|S.oester Börde|So.ester Börde|Soe.ster Börde|Soes.ter Börde|Soest.er Börde|Soeste.r Börde|Soester. Börde|Soester .Börde|Soester B.örde|Soester Bö.rde|Soester Bör.de|Soester Börd.e|Sester Börde|Soster Börde|Soeter Börde|Soeser Börde|Soestr Börde|Soeste Börde|SoesterBörde|Soester örde|Soester Brde|Soester Böde|Soester Böre': TopCategory.EXZERPT,  # Wörterbuch Soester Börde → 4920 (0.41%)
    r'[Kk]{2}[Ww]b': TopCategory.EXZERPT,  # KkWb → 4738 (0.39%)
    r'.andebeck|S.ndebeck|Sa.debeck|San.ebeck|Sand.beck|Sande.eck|Sandeb.ck|Sandebe.k|Sandebec.|S.andebeck|Sa.ndebeck|San.debeck|Sand.ebeck|Sande.beck|Sandeb.eck|Sandebe.ck|Sandebec.k|Sndebeck|Sadebeck|Sanebeck|Sandbeck|Sandeeck|Sandebck|Sandebek': TopCategory.LAUTSCHRIFT,  # Sandebeck → 4701 (0.39%)
    r' .rchiv|A.chiv|Ar.hiv|Arc.iv|Arch.v|Archi.|A.rchiv|Ar.chiv|Arc.hiv|Arch.iv|Archi.v|Achiv|Arhiv|Arciv|Archv': TopCategory.EXZERPT,  # Archiv für westfälische Volkskunde → 4426 (0.37%)
    r'Rheine V': TopCategory.LAUTSCHRIFT,  # Rheine August Vollmer → 4224 (0.35%)
    r' [0O]el[. ]|Drewer.+?Oel': TopCategory.WORTSCHATZ,  # 4033 (0.33%)
    r'.chwelm|S.hwelm|Sc.welm|Sch.elm|Schw.lm|Schwe.m|Schwel.|S.chwelm|Sc.hwelm|Sch.welm|Schw.elm|Schwe.lm|Schwel.m|Shwelm|Scwelm|Schelm|Schwlm|Schwem': TopCategory.LAUTSCHRIFT,  # Schwelm → 3871 (0.32%)
    r'.olthaus Mskr|H.lthaus Mskr|Ho.thaus Mskr|Hol.haus Mskr|Holt.aus Mskr|Holth.us Mskr|Holtha.s Mskr|Holthau. Mskr|Holthaus.Mskr|Holthaus .skr|Holthaus M.kr|Holthaus Ms.r|Holthaus Msk.|H.olthaus Mskr|Ho.lthaus Mskr|Hol.thaus Mskr|Holt.haus Mskr|Holth.aus Mskr|Holtha.us Mskr|Holthau.s Mskr|Holthaus. Mskr|Holthaus .Mskr|Holthaus M.skr|Holthaus Ms.kr|Holthaus Msk.r|Hlthaus Mskr|Hothaus Mskr|Holhaus Mskr|Holtaus Mskr|Holthus Mskr|Holthas Mskr|Holthau Mskr|HolthausMskr|Holthaus skr|Holthaus Mkr|Holthaus Msr': TopCategory.EXZERPT,  # Holthaus Mskr → 3730 (0.31%)
    r'Wix|h wix': TopCategory.LAUTSCHRIFT,  # Gütersloh Wix
    r'.aumann|K.umann|Ka.mann|Kau.ann|Kaum.nn|Kauma.n|Kauman.|K.aumann|Ka.umann|Kau.mann|Kaum.ann|Kauma.nn|Kauman.n|Kumann|Kamann|Kauann|Kaumnn|Kauman': TopCategory.WORTSCHATZ,  # Kaumann → 2927 (0.24%)
    r'.chmitz|S.hmitz|Sc.mitz|Sch.itz|Schm.tz|Schmi.z|Schmit.|S.chmitz|Sc.hmitz|Sch.mitz|Schm.itz|Schmi.tz|Schmit.z|Shmitz|Scmitz|Schitz|Schmtz|Schmiz': TopCategory.WORTSCHATZ,  # Schmitz → 2843 (0.23%)
    r'Elsey': TopCategory.LAUTSCHRIFT,  # Iserlohn Elsey → 2835 (0.23%)
    r'.ahmede|R.hmede|Ra.mede|Rah.ede|Rahm.de|Rahme.e|Rahmed.|R.ahmede|Ra.hmede|Rah.mede|Rahm.ede|Rahme.de|Rahmed.e|Rhmede|Ramede|Rahede|Rahmde|Rahmee': TopCategory.WORTSCHATZ,  # Rahmede → 2601 (0.21%)
    r'.serlohn|I.erlohn|Is.rlohn|Ise.lohn|Iser.ohn|Iserl.hn|Iserlo.n|Iserloh.|I.serlohn|Is.erlohn|Ise.rlohn|Iser.lohn|Iserl.ohn|Iserlo.hn|Iserloh.n|Ierlohn|Isrlohn|Iselohn|Iserohn|Iserlhn|Iserlon': TopCategory.LAUTSCHRIFT,  # Iserlohn → 2424 (0.20%)
    r'.ltena|A.tena|Al.ena|Alt.na|Alte.a|Alten.|A.ltena|Al.tena|Alt.ena|Alte.na|Alten.a|Atena|Alena|Altna|Altea': TopCategory.LAUTSCHRIFT,  # Altena → 2308 (0.19%)
    r'.öhner|G.hner|Gö.ner|Göh.er|Göhn.r|Göhne.|G.öhner|Gö.hner|Göh.ner|Göhn.er|Göhne.r|Ghner|Göner|Göher|Göhnr': TopCategory.WORTSCHATZ,  # Göhner in Hfd Go → 2289 (0.19%)
    r'.üller|M[^ö]ller|Mü.ler|Mül.er|Müll.r|Mülle.|M.üller|Mü.ller|Mül.ler|Müll.er|Mülle.r|Mller|Müler|Müler|Müllr': TopCategory.WORTSCHATZ,  # K. A. Müller → 2195 (0.18%)
    r'.chröder|S.hröder|Sc.röder|Sch.öder|Schr.der|Schrö.er|Schröd.r|Schröde.|S.chröder|Sc.hröder|Sch.röder|Schr.öder|Schrö.der|Schröd.er|Schröde.r|Shröder|Scröder|Schöder|Schrder|Schröer|Schrödr': TopCategory.WORTSCHATZ,  # Schröder → 2169 (0.18%)
    r'.essum|W.ssum|We.sum|Wes.um|Wess.m|Wessu.|W.essum|We.ssum|Wes.sum|Wess.um|Wessu.m|Wssum|Wesum|Wesum|Wessm': TopCategory.LAUTSCHRIFT,  # Wessum → 2141 (0.18%)
    r'Drenst': TopCategory.LAUTSCHRIFT,  # Drenstein → 2087 (0.17%)
    r'.öpper|K.pper|Kö.per|Köp.er|Köpp.r|Köppe.|K.öpper|Kö.pper|Köp.per|Köpp.er|Köppe.r|Kpper|Köper|Köper|Köppr': TopCategory.WORTSCHATZ,  # Köpper in Dortmund → 1994 (0.16%)
    r'Ly.a|Lyr.': TopCategory.EXZERPT,  # Lyra Osnabrück → 1824 (0.15%)
    r'.choppe|S.hoppe|Sc.oppe|Sch[^e]ppe|Scho.pe|Schop.e|Schopp.|S.choppe|Sc.hoppe|Sch.oppe|Scho.ppe|Schop.pe|Schopp.e|Shoppe|Scoppe|Schppe|Schope|Schope': TopCategory.WORTSCHATZ,  # Schoppe → 1522 (0.13%)
    r'.agenf|W.genf|Wa.enf|Wag.nf|Wage.f|W.agenf|Wa.genf|Wag.enf|Wage.nf|Wagen.f|Wgenf|Waenf|Wagnf|Wagef': TopCategory.EXZERPT,  # Wagenfeld → 1642 (0.14%)
    r'.chepers|S.hepers|Sc.epers|Sch.pers|Sche.ers|Schep.rs|Schepe[^l]s|S.chepers|Sc.hepers|Sch.epers|Sche.pers|Schep.ers|Schepe.rs|Scheper.s|Shepers|Scepers|Schpers|Scheers|Scheprs|Schepes': TopCategory.WORTSCHATZ,  # Schepers → 1487 (0.12%)
    r'.akers|R.kers|Ra.ers|Rak.rs|Rake.s|Raker.|R.akers|Ra.kers|Rak.ers|Rake.rs|Raker.s|Rkers|Raers|Rakrs|Rakes': TopCategory.WORTSCHATZ,  # Rakers → 1474 (0.12%)
    r'[^d].archiv|r.rchiv|ra.chiv|rar.hiv|rarc.iv|rarch.v|rarchi.|r.archiv|ra.rchiv|rar.chiv|rarc.hiv|rarch.iv|rarchi.v|rrchiv|rachiv|rarhiv|rarciv|rarchv|[^d][^r]archiv|.RCHIV|A.CHIV|AR.HIV|ARC.IV|ARCH.V|ARCHI.|A.RCHIV|AR.CHIV|ARC.HIV|ARCH.IV|ARCHI.V|ACHIV|ARHIV|ARCIV|ARCHV': TopCategory.EXZERPT,  # Westfälisches Sprichwörterarchiv → 1442 (0.12%)
    r'Arens ': TopCategory.LAUTSCHRIFT,  # Arens → 1353 (0.11%)
    r'.lesken|B.esken|Bl.sken|Ble.ken|Bles.en|Blesk.n|Bleske.|B.lesken|Bl.esken|Ble.sken|Bles.ken|Blesk.en|Bleske.n|Besken|Blsken|Bleken|Blesen|Bleskn': TopCategory.WORTSCHATZ,  # Blesken → 1299 (0.11%)
    r'.regory|G.egory|Gr.gory|Gre.ory|Greg.ry|Grego.y|Gregor.|G.regory|Gr.egory|Gre.gory|Greg.ory|Grego.ry|Gregor.y|Gegory|Grgory|Greory|Gregry|Gregoy': TopCategory.LAUTSCHRIFT,  # Gregory → 1234 (0.10%)
    r'.eddigen|W.ddigen|We.digen|Wed.igen|Wedd.gen|Weddi.en|Weddig.n|Weddige.|W.eddigen|We.ddigen|Wed.digen|Wedd.igen|Weddi.gen|Weddig.en|Weddige.n|Wddigen|Wedigen|Wedigen|Weddgen|Weddien|Weddign': TopCategory.EXZERPT,  # Weddigen → 1188 (0.10%)
    r'Blesken': TopCategory.WORTSCHATZ,  # A. H. Blesken → 1093 (0.09%)
    r'.üschede W|M.schede W|Mü.chede W|Müs.hede W|Müsc.ede W|Müsch.de W|Müsche.e W|Müsched. W|Müschede.W|Müschede .|M.üschede W|Mü.schede W|Müs.chede W|Müsc.hede W|Müsch.ede W|Müsche.de W|Müsched.e W|Müschede. W|Müschede .W|Mschede W|Müchede W|Müshede W|Müscede W|Müschde W|Müschee W|Müsched W|MüschedeW': TopCategory.LAUTSCHRIFT,  # Müschede → 1071 (0.09%)
    r'Nd\.Kbl\.|.d\. Kbl\.|N.\. Kbl\.|Nd.. Kbl\.|Nd\. Kbl\.|Nd\..Kbl\.|Nd\. .bl\.|Nd\. K.l\.|Nd\. Kb.\.|Nd\. Kbl..|Nd\. Kbl\.|N.d\. Kbl\.|Nd.\. Kbl\.|Nd\.. Kbl\.|Nd\.. Kbl\.|Nd\. .Kbl\.|Nd\. K.bl\.|Nd\. Kb.l\.|Nd\. Kbl.\.|Nd\. Kbl\..': TopCategory.EXZERPT,  # Nd. Kbl. → 1039 (0.09%)
    r'.Berger\)|\(.erger\)|\(B.rger\)|\(Be.ger\)|\(Ber.er\)|\(Berg.r\)|\(Berge.\)|\(Berger.|\(.Berger\)|\(B.erger\)|\(Be.rger\)|\(Ber.ger\)|\(Berg.er\)|\(Berge.r\)|\(Berger.\)|\(erger\)|\(Brger\)|\(Beger\)|\(Berer\)|\(Bergr\)|\(Berge\)': TopCategory.WORTSCHATZ,  # Berger → 1025 (0.098)
    r'Westph': TopCategory.EXZERPT,  # Westph. Mag. → 812 (0.07%)
    r'.ahlmann|B.hlmann|Ba.lmann|Bah.mann|Bahl.ann|Bahlm.nn|Bahlma.n|Bahlman.|B.ahlmann|Ba.hlmann|Bah.lmann|Bahl.mann|Bahlm.ann|Bahlma.nn|Bahlman.n|Bhlmann|Balmann|Bahmann|Bahlann|Bahlmnn|Bahlman': TopCategory.EXZERPT,  # Bahlmann → 778 (0.06%)
    r'Linden': TopCategory.WORTSCHATZ,  # Linden, G. Pott → 778 (0.06%)
    r'.rundsteinheim|G.undsteinheim|Gr.ndsteinheim|Gru.dsteinheim|Grun.steinheim|Grund.teinheim|Grunds.einheim|Grundst.inheim|Grundste.nheim|Grundstei.heim|Grundstein.eim|Grundsteinh.im|Grundsteinhe.m|Grundsteinhei.|G.rundsteinheim|Gr.undsteinheim|Gru.ndsteinheim|Grun.dsteinheim|Grund.steinheim|Grunds.teinheim|Grundst.einheim|Grundste.inheim|Grundstei.nheim|Grundstein.heim|Grundsteinh.eim|Grundsteinhe.im|Grundsteinhei.m|Gundsteinheim|Grndsteinheim|Grudsteinheim|Grunsteinheim|Grundteinheim|Grundseinheim|Grundstinheim|Grundstenheim|Grundsteiheim|Grundsteineim|Grundsteinhim|Grundsteinhem': TopCategory.WORTSCHATZ,  # Grundsteinheim, Lehrer Pagendarm → 681 (0.06%)
    r'.unold|H.nold|Hu.old|Hun.ld|Huno.d|Hunol.|H.unold|Hu.nold|Hun.old|Huno.ld|Hunol.d|Hnold|Huold|Hunld|Hunod': TopCategory.EXZERPT,  # Hunold → 675 (0.06%)
    r'Kleinn|K.einn|Kl.inn|Kle.nn|Klei.n|K.leinn|Kl.einn|Kle.inn|Klei.nn|Keinn|Klinn|Klenn': TopCategory.WORTSCHATZ,  # Kleinn → 651 (0.05%)
    r'.latenauSp|P.atenauSp|Pl.tenauSp|Pla.enauSp|Plat.nauSp|Plate.auSp|Platen.uSp|Platena.Sp|Platenau.p|PlatenauS.|P.latenauSp|Pl.atenauSp|Pla.tenauSp|Plat.enauSp|Plate.nauSp|Platen.auSp|Platena.uSp|Platenau.Sp|PlatenauS.p|PatenauSp|PltenauSp|PlaenauSp|PlatnauSp|PlateauSp|PlatenuSp|PlatenaSp|Platenaup': TopCategory.EXZERPT,  # PlatenauSp → 644 (0.05%)
    r'Endorf': TopCategory.LAUTSCHRIFT,  # Endorf → 636 (0.05%)
    r'P.ckers|Pi.kers|Pic.ers|Pick.rs|Picke.s|Picker.|P.ickers|Pi.ckers|Pic.kers|Pick.ers|Picke.rs|Picker.s|Pckers|Pikers|Picers|Pickrs|Pickes': TopCategory.WORTSCHATZ,  # Pickers → 628 (0.05%)
    r'.athen Schönhoff|L.then Schönhoff|La.hen Schönhoff|Lat.en Schönhoff|Lath.n Schönhoff|Lathe. Schönhoff|Lathen.Schönhoff|Lathen .chönhoff|Lathen S.hönhoff|Lathen Sc.önhoff|Lathen Sch.nhoff|Lathen Schö.hoff|Lathen Schön.off|Lathen Schönh.ff|Lathen Schönho.f|Lathen Schönhof.|L.athen Schönhoff|La.then Schönhoff|Lat.hen Schönhoff|Lath.en Schönhoff|Lathe.n Schönhoff|Lathen. Schönhoff|Lathen .Schönhoff|Lathen S.chönhoff|Lathen Sc.hönhoff|Lathen Sch.önhoff|Lathen Schö.nhoff|Lathen Schön.hoff|Lathen Schönh.off|Lathen Schönho.ff|Lathen Schönhof.f|Lthen Schönhoff|Lahen Schönhoff|Laten Schönhoff|Lathn Schönhoff|Lathe Schönhoff|LathenSchönhoff|Lathen chönhoff|Lathen Shönhoff|Lathen Scönhoff|Lathen Schnhoff|Lathen Schöhoff|Lathen Schönoff|Lathen Schönhff|Lathen Schönhof': TopCategory.LAUTSCHRIFT,  # Lathen Schönhoff → 531 (0.04%)
    r'(?:.ochum|B.chum|Bo.hum|Boc.um|Boch.m|Bochu.|B.ochum|Bo.chum|Boc.hum|Boch.um|Bochu.m|Bchum|Bohum|Bocum|Bochm).{0,4}(?:.aer|L.er|La.r|Lae.|L.aer|La.er|Lae.r|Ler|Lar)': TopCategory.EXZERPT,  # Bochum - Laer → 528 (0.04%)
    r'.üld: Volk|B.ld: Volk|Bü.d: Volk|Bül.: Volk|Büld. Volk|Büld:.Volk|Büld: .olk|Büld: V.lk|Büld: Vo.k|Büld: Vol.|B.üld: Volk|Bü.ld: Volk|Bül.d: Volk|Büld.: Volk|Büld:. Volk|Büld: .Volk|Büld: V.olk|Büld: Vo.lk|Büld: Vol.k|Bld: Volk|Büd: Volk|Bül: Volk|Büld Volk|Büld:Volk|Büld: olk|Büld: Vlk|Büld: Vok': TopCategory.EXZERPT,  # Büld: Volku. Spr. → 507 (0.04%)
    r'.aub Sp|R.ub Sp|Ra.b Sp|Rau. Sp|Raub.Sp|Raub .p|Raub S.|R.aub Sp|Ra.ub Sp|Rau.b Sp|Raub. Sp|Raub .Sp|Raub S.p|Rub Sp|Rab Sp|Rau Sp|RaubSp|Raub p': TopCategory.EXZERPT,  # RaubSp → 342 (0.03%)
    r'.kten Minden|A.ten Minden|Ak.en Minden|Akt.n Minden|Akte. Minden|Akten.Minden|Akten .inden|Akten M.nden|Akten Mi.den|Akten Min.en|Akten Mind.n|Akten Minde.|A.kten Minden|Ak.ten Minden|Akt.en Minden|Akte.n Minden|Akten. Minden|Akten .Minden|Akten M.inden|Akten Mi.nden|Akten Min.den|Akten Mind.en|Akten Minde.n|Aten Minden|Aken Minden|Aktn Minden|Akte Minden|AktenMinden|Akten inden|Akten Mnden|Akten Miden|Akten Minen|Akten Mindn': TopCategory.EXZERPT,  # Reg. Akten Minden → 444 (0.04%)
    r'Prov\.': TopCategory.EXZERPT,  # Ber. Westf. Prov. Ver. → 347 (0.03%)
    r'Gescher': TopCategory.LAUTSCHRIFT,  # Gescher → 186 (0.02%)
    r'.hr\. Koch|C.r\. Koch|Ch.\. Koch|Chr.. Koch|Chr\. Koch|Chr\..Koch|Chr\. .och|Chr\. K.ch|Chr\. Ko.h|Chr\. Koc.|C.hr\. Koch|Ch.r\. Koch|Chr.\. Koch|Chr\.. Koch|Chr\.. Koch|Chr\. .Koch|Chr\. K.och|Chr\. Ko.ch|Chr\. Koc.h|Cr\. Koch|Ch\. Koch|Chr. Koch|Chr\ Koch|Chr\.Koch|Chr\. och|Chr\. Kch|Chr\. Koh': TopCategory.WORTSCHATZ,  # Christine Koch → 174 (0.01%)
}

In [ ]:
def generate_regex(word: str):
    regex = r''
    for i in range(len(word)):
        regex += word[:i] + r'.' + word[i+1:] + '|'
    for i in range(1, len(word)):
        regex += word[:i] + r'.' + word[i:] + '|'
    for i in range(1, len(word)-1):
        regex += word[:i] + word[i+1:] + '|'
    print(regex[:-1])

generate_regex(r'Hunold')

.unold|H.nold|Hu.old|Hun.ld|Huno.d|Hunol.|H.unold|Hu.nold|Hun.old|Huno.ld|Hunol.d|Hnold|Huold|Hunld|Hunod


In [ ]:
def filter_patterns():
    out_counter = 0
    tot_counter = 0
    for text in texts:
        tot_counter += 1
        out = False
        for pattern in classification:
            if not out and len(re.findall(pattern, text[1])):
                out = True
                out_counter += 1
        #if not out:
        #    print(text)
        #if tot_counter % 100000 == 0:
        #    break
    print(f'{out_counter}/{tot_counter} ({(100*out_counter/tot_counter):.2f}%)')

filter_patterns()

630922/1211661 (52.07%)
